# Bronze Layer v2 — Hourly Scheduled Load: Source Blob → Charging Sessions

**Day 3 | Scheduled via Databricks Job — runs every hour automatically.**

Reads the current hour's CSV files from the instructor-provided source blob and copies them into the Bronze Volume, preserving the exact source directory structure. No transformation — files land in Bronze exactly as they are in the source.

---

### Source layout
```
wasbs://source@dataenggdailystorage.blob.core.windows.net/
  └── realtime/
        └── charging_sessions/
              └── YYYY/MM/DD/HH/
                    └── sessions_YYYYMMDD_HHMM.csv
```

### Bronze target (mirrors source exactly)
```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
  └── realtime/
        └── charging_sessions/
              └── YYYY/MM/DD/HH/
                    └── sessions_YYYYMMDD_HHMM.csv
```

> **Bronze folder creation:** The `YYYY/MM/DD/HH/` directory hierarchy is **never pre-created manually**. `dbutils.fs.cp` creates each level automatically when it copies the first file into that path — same behaviour in both v1 and v2.

---

### What changed from v1 → v2

| | v1 | v2 |
|---|---|---|
| **Partition selection** | Manually set `LOAD_YEAR`, `LOAD_MONTH`, `LOAD_DAY`, `LOAD_HOUR` in Cell 2 | Auto-computed from `datetime.now(UTC)` at runtime |
| **Trigger** | Run notebook manually | Databricks Job — cron `0 * * * *` (top of every hour) |
| **Full load** | `LOAD_MODE = "full"` | `FULL_LOAD_OVERRIDE = True` (one-off flag, reset to False after) |
| **Bronze folder creation** | Auto-created by `dbutils.fs.cp` during copy | Same — `YYYY/MM/DD/HH/` created automatically on Bronze Volume |
| **Missing source hour** | Crashes with `Path does not exist` if source folder absent | Checks source folder exists first — exits cleanly if source has no data for that hour |
| **Failure alerting** | None | Job marks run Failed + sends email if copy errors occur |

## Cell 1 — Authenticate to source blob

Reads the storage account name, container name, and SAS token from Key Vault via the `kv-ev-scope` Databricks secret scope. Nothing is hardcoded.

**Must run every session** — `spark.conf.set` registers the SAS credential for the `wasbs://` driver. This config does not persist across cluster restarts, so always run this cell first.

- `source-storage-account` → blob storage account name
- `source-container` → container name (`source`)
- `source-sas-token` → SAS token with `sp=rl` (read + list) permissions
- `SOURCE_ROOT` → base `wasbs://` URL used by all subsequent cells

In [ ]:
from datetime import datetime, timezone

SCOPE = "kv-ev-scope"

STORAGE_ACCOUNT = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
CONTAINER       = dbutils.secrets.get(scope=SCOPE, key="source-container")
SAS_TOKEN       = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")

spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)

SOURCE_ROOT = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"

print(f"Storage account : {STORAGE_ACCOUNT}")
print(f"Container       : {CONTAINER}")
print(f"SAS token       : [REDACTED]")
print(f"Source root     : {SOURCE_ROOT}")
print("Source blob authenticated — OK")

## Cell 2 — Resolve the 3-hour look-back window from system clock

**This cell never needs manual editing when running as a scheduled Job.**

The Job fires at the top of every hour (cron `0 * * * *`). At that point the **previous hour** has fully completed — source system finishes writing before the next hour boundary. We subtract 1 hour from `datetime.now(UTC)` as the primary target.

**Why 3-hour window?**
Source data can arrive late (network delay, upstream pipeline lag). Instead of loading only the previous hour and missing late data, we check the **3 most recent completed hours** on every run:

```
Job fires at 09:00 UTC → checks hours: 08, 07, 06
  - If Bronze already has data for that hour  → SKIP (already loaded)
  - If Bronze has no data for that hour       → COPY from source
```

This means:
- A late-arriving file for `07/` will be picked up on the `08:00`, `09:00`, or `10:00` run — whichever is first to find it
- No manual backfill needed for up to 3 hours of late data
- Already-loaded hours are skipped efficiently — no duplicate writes

**`FULL_LOAD_OVERRIDE`** — set to `True` only for a one-off full historical load. Reset to `False` before the next scheduled run.

| `FULL_LOAD_OVERRIDE` | What gets loaded |
|---|---|
| `False` (default) | 3-hour look-back window — skips hours already in Bronze |
| `True` | Everything under `realtime/charging_sessions/` — all history |

In [ ]:
from datetime import datetime, timezone, timedelta

# Set to True only for a one-off full historical load.
# Leave False for all normal hourly scheduled runs.
FULL_LOAD_OVERRIDE = False

# Number of completed hours to check on every run.
# Each hour is skipped if Bronze already has data — only missing hours are copied.
LOOKBACK_HOURS = 3

BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
BASE_SUBPATH  = "realtime/charging_sessions"

now = datetime.now(timezone.utc)
print(f"Job fire time (UTC) : {now.strftime('%Y-%m-%d %H:%M:%S')} UTC")

if FULL_LOAD_OVERRIDE:
    # Full load — single pass over entire history
    window = [{"source_path": f"{SOURCE_ROOT}/{BASE_SUBPATH}/",
               "bronze_path": f"{BRONZE_VOLUME}/{BASE_SUBPATH}/",
               "label": "FULL (override)"}]
else:
    # Build look-back window: previous LOOKBACK_HOURS completed hours
    # e.g. job fires 09:00 → window = [08, 07, 06]
    window = []
    for offset in range(1, LOOKBACK_HOURS + 1):
        target     = now - timedelta(hours=offset)
        partition  = target.strftime("%Y/%m/%d/%H")
        window.append({
            "source_path": f"{SOURCE_ROOT}/{BASE_SUBPATH}/{partition}/",
            "bronze_path": f"{BRONZE_VOLUME}/{BASE_SUBPATH}/{partition}/",
            "label": f"INCREMENTAL — {partition}"
        })

print(f"\nLook-back window ({len(window)} hour(s)):")
for w in window:
    print(f"  {w['label']}")

## Cell 3 — Filter window: skip hours already in Bronze, skip hours missing from source

For each hour in the look-back window, two checks run before any copy is attempted:

**Check 1 — Already in Bronze?**
If the Bronze destination folder exists and contains at least one file, that hour was already loaded successfully on a previous run. Skip it — no duplicate copy needed.

**Check 2 — Source folder exists?**
If the source blob has no folder for that hour, the upstream system has not written data yet (or skipped that hour entirely). Skip it and move on — the next run's window will still include this hour if it falls within 3 hours.

Only hours that pass both checks proceed to the copy step.

```
Example — job fires 09:00 UTC:
  Hour 08  → Bronze has data   → SKIP (already loaded)
  Hour 07  → Source missing    → SKIP (not arrived yet — will retry next run)
  Hour 06  → Both exist        → COPY
```

In [ ]:
def folder_exists(path):
    try:
        files = dbutils.fs.ls(path)
        return len(files) > 0
    except Exception:
        return False

hours_to_copy = []

for w in window:
    label       = w["label"]
    source_path = w["source_path"]
    bronze_path = w["bronze_path"]

    # Check 1 — already loaded in Bronze?
    if not FULL_LOAD_OVERRIDE and folder_exists(bronze_path):
        print(f"  SKIP (already in Bronze) : {label}")
        continue

    # Check 2 — source folder exists?
    if not folder_exists(source_path):
        print(f"  SKIP (source not found)  : {label}")
        continue

    print(f"  QUEUE for copy           : {label}")
    hours_to_copy.append(w)

print(f"\nHours queued for copy: {len(hours_to_copy)}")

if not hours_to_copy:
    msg = "Nothing to copy — all hours in window already loaded or source data not yet available."
    print(f"\nINFO: {msg}")
    dbutils.notebook.exit(msg)

## Cell 4 — List source files for each queued hour

Recursively lists all files under each source path that passed the checks in Cell 3.

For a normal scheduled run each queued hour typically contains one CSV file: `sessions_YYYYMMDD_HHMM.csv`. For a full load override it recurses all year/month/day/hour subdirectories.

In [ ]:
def list_files_recursive(path):
    items = dbutils.fs.ls(path)
    files = []
    for item in items:
        if item.isDir():
            files.extend(list_files_recursive(item.path))
        else:
            files.append(item)
    return files

# Attach source file list to each queued hour
for w in hours_to_copy:
    w["source_files"] = list_files_recursive(w["source_path"])
    print(f"\n{w['label']} — {len(w['source_files'])} file(s):")
    for f in w["source_files"]:
        print(f"  {f.path}  [{round(f.size/1024, 1)} KB]")

## Cell 5 — Copy files to Bronze Volume for each queued hour

Copies files from source to Bronze for every hour that was queued in Cell 3.

**How the destination path is built:**
```
source_file.path = wasbs://.../charging_sessions/2026/07/06/07/sessions_20260706_0700.csv
source_path      = wasbs://.../charging_sessions/2026/07/06/07/
relative_path    = sessions_20260706_0700.csv
dest_path        = /Volumes/.../charging_sessions/2026/07/06/07/sessions_20260706_0700.csv
```

The `YYYY/MM/DD/HH/` folder hierarchy on Bronze is **created automatically** by `dbutils.fs.cp` — no pre-creation needed.

**Idempotency:** overwriting an existing file is safe — re-running produces the same result.

**On failure:** any copy error raises an exception at the end — Job marks run as Failed and sends alert.

In [ ]:
all_copied  = []
all_skipped = []

for w in hours_to_copy:
    source_path  = w["source_path"]
    bronze_path  = w["bronze_path"]
    source_files = w["source_files"]
    label        = w["label"]

    copied  = []
    skipped = []

    print(f"\n--- {label} ---")
    for file_info in source_files:
        relative_path = file_info.path.replace(source_path, "")
        dest_path     = bronze_path + relative_path
        try:
            dbutils.fs.cp(file_info.path, dest_path)
            copied.append(dest_path)
            print(f"  COPIED  {relative_path}")
        except Exception as e:
            skipped.append((file_info.path, str(e)))
            print(f"  FAILED  {relative_path} — {e}")

    print(f"  Result: {len(copied)} copied, {len(skipped)} failed")
    all_copied.extend(copied)
    all_skipped.extend(skipped)

print(f"\nTotal: {len(all_copied)} copied, {len(all_skipped)} failed across {len(hours_to_copy)} hour(s)")

if all_skipped:
    raise Exception(f"{len(all_skipped)} file(s) failed to copy — check output above.")

## Cell 6 — Verify files in Bronze Volume for each copied hour

For each hour that was copied, lists files at the Bronze destination and asserts count matches source. If any assertion fails the Job run is marked Failed and an alert fires.

In [ ]:
for w in hours_to_copy:
    bronze_path  = w["bronze_path"]
    source_files = w["source_files"]
    label        = w["label"]

    bronze_files = list_files_recursive(bronze_path)
    print(f"\n{label} — Bronze: {len(bronze_files)} file(s)")
    for f in bronze_files:
        print(f"  {f.path}  [{round(f.size/1024, 1)} KB]")

    assert len(bronze_files) == len(source_files), (
        f"Count mismatch for {label} — source: {len(source_files)}, bronze: {len(bronze_files)}"
    )
    print(f"  Verification passed.")

print("\nAll hours verified.")

## Cell 7 — Read sample file and inspect schema

Reads the first CSV from the first copied hour into Spark — lightweight sanity check to confirm the file is readable and columns look as expected. Silver layer will enforce explicit schema; `inferSchema` here is for inspection only.

In [ ]:
first_hour   = hours_to_copy[0]
bronze_files = list_files_recursive(first_hour["bronze_path"])
sample_file  = bronze_files[0].path

print(f"Reading sample from: {first_hour['label']}")
print(f"File: {sample_file}")

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(sample_file)

print(f"Row count : {df.count():,}")
print(f"Columns   : {len(df.columns)}")
df.printSchema()
display(df.limit(5))

## Cell 8 — Run summary

Prints a final summary of the run: timestamp, load mode, paths, file counts, and any failures.

When running as a scheduled Job, this output is visible in:
**Databricks → Workflows → `job_bronze_charging_sessions_hourly` → Run history → this run → Output**

---

**What comes next:** This notebook lands raw CSVs in the Bronze Volume. The Silver layer notebook (Day 7) reads from `/Volumes/.../bronze-volume/realtime/charging_sessions/`, applies explicit schema, deduplicates by `session_id`, and writes Delta.

In [ ]:
print("=" * 60)
print("BRONZE BLOB MIGRATION v2 — HOURLY RUN SUMMARY")
print("=" * 60)
print(f"Job fire time (UTC) : {now.strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"Look-back window    : {LOOKBACK_HOURS} hour(s)")
print(f"Hours in window     : {len(window)}")
print(f"Hours copied        : {len(hours_to_copy)}")
print(f"Hours skipped       : {len(window) - len(hours_to_copy)}")
print(f"Total files copied  : {len(all_copied)}")
print(f"Total files failed  : {len(all_skipped)}")
print()
for w in window:
    status = "COPIED" if w in hours_to_copy else "SKIPPED"
    print(f"  [{status}] {w['label']}")
print("=" * 60)
print("Next step: Silver layer reads from Bronze Volume and writes Delta.")